# Word2Vec Implementation From Scratch - Complete Notebook Flow

## Goal

Convert SMS messages into numerical vectors using Word2Vec and use those vectors to classify messages as Spam or Ham.

---

# STEP 1: Import Libraries

Import all required libraries.

```python
import pandas as pd
import numpy as np
import re
import nltk
import gensim
```

Purpose:
- Data handling → Pandas
- Numerical operations → NumPy
- Text cleaning → Regex
- NLP preprocessing → NLTK
- Word2Vec → Gensim

---

# STEP 2: Load Pretrained Google Word2Vec Model

```python
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')
```

Purpose:
- Explore an already trained Word2Vec model.
- Understand how word embeddings work.

Example:

```python
wv['king']
```

Returns a 300-dimensional vector.

---

# STEP 3: Check Similar Words

```python
wv.most_similar('good')
```

Purpose:
- Observe semantic relationships learned by Word2Vec.

Example:

```text
good → great, nice, excellent ...
```

---

# STEP 4: Load SMS Spam Dataset

```python
messages = pd.read_csv(
    'SMSSpamCollection.csv',
    sep='\t',
    names=['label','message']
)
```

Dataset contains:

| Label | Message |
|---------|---------|
| ham | Normal message |
| spam | Spam message |

---

# STEP 5: Data Cleaning

Create an empty corpus.

```python
corpus = []
```

Loop through every message.

For each message:

### Remove Special Characters

```python
review = re.sub('[^a-zA-Z]',' ',message)
```

### Convert to Lowercase

```python
review = review.lower()
```

### Split into Words

```python
review.split()
```

### Lemmatization

```python
WordNetLemmatizer()
```

Example:

```text
running → run
cars → car
```

### Join Back

```python
' '.join(review)
```

Store inside:

```python
corpus.append(review)
```

Result:

```python
corpus
```

contains cleaned text.

---

# STEP 6: Sentence Tokenization

Convert each cleaned sentence into a list of words.

```python
from gensim.utils import simple_preprocess
```

```python
words = []

for sentence in corpus:
    words.append(simple_preprocess(sentence))
```

Example:

Before:

```text
i love machine learning
```

After:

```python
['i','love','machine','learning']
```

Result:

```python
words
```

becomes:

```python
[
 ['go','until','jurong'],
 ['ok','lar'],
 ...
]
```

---

# STEP 7: Train Word2Vec Model

```python
model = gensim.models.Word2Vec(
    words,
    vector_size=100,
    window=5,
    min_count=1
)
```

Purpose:
- Learn word embeddings from the SMS dataset itself.

Output:
- Every word gets a 100-dimensional vector.

---

# STEP 8: Inspect Vocabulary

```python
model.wv.index_to_key
```

Purpose:
- View all learned words.

---

# STEP 9: Get Word Vector

```python
model.wv['good']
```

Purpose:
- Retrieve embedding of a word.

Output:

```text
[100 numerical values]
```

---

# STEP 10: Find Similar Words

```python
model.wv.most_similar('good')
```

Purpose:
- Check whether Word2Vec learned semantic relationships.

---

# STEP 11: Create Sentence Embeddings

Problem:

Word2Vec creates vectors for words.

Machine Learning model requires one vector per message.

Solution:

Average all word vectors.

---

# STEP 12: Define Average Word2Vec Function

```python
def avg_word2vec(doc):
```

Steps:

1. Take all words in a sentence.
2. Fetch vector for each word.
3. Sum vectors.
4. Divide by number of words.

Result:

One fixed-length vector for one message.

---

# STEP 13: Generate Feature Matrix X

```python
X = []
```

Loop through all messages:

```python
for i in words:
    X.append(avg_word2vec(i))
```

Convert into array.

```python
X = np.array(X)
```

Result:

```python
X.shape
```

Approximately:

```text
(5569,100)
```

Meaning:

5569 messages

100 features per message

---

# STEP 14: Encode Labels

Convert:

```text
ham  → 0
spam → 1
```

Using:

```python
pd.get_dummies(messages['label'])
```

Store in:

```python
y
```

---

# STEP 15: Remove Empty Rows

Some messages become empty after preprocessing.

Those rows cannot produce Word2Vec vectors.

Remove corresponding rows from:

```python
X
y
```

Purpose:

Keep dimensions aligned.

---

# STEP 16: Convert Features to DataFrame

```python
df = pd.DataFrame(X)
```

Purpose:
- Easier inspection and debugging.

---

# STEP 17: Train-Test Split

```python
from sklearn.model_selection import train_test_split
```

```python
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0
)
```

Purpose:
- Training data → model learning
- Testing data → evaluation

---

# STEP 18: Train Random Forest

```python
from sklearn.ensemble import RandomForestClassifier
```

```python
classifier = RandomForestClassifier()
```

```python
classifier.fit(X_train,y_train)
```

Purpose:
- Learn spam vs ham classification.

---

# STEP 19: Hyperparameter Tuning

Create parameter grid.

```python
para = {
    'n_estimators':[100,200,300,400,500],
    'max_depth':[10,20,30,40,50]
}
```

Apply GridSearchCV.

```python
GridSearchCV(
    estimator=classifier,
    param_grid=para,
    cv=5,
    n_jobs=-1
)
```

Meaning:

25 parameter combinations

5-fold cross validation

Total:

```text
125 model trainings
```

---

# STEP 20: Train Best Model

```python
model.fit(X_train,y_train)
```

GridSearch finds best parameters.

---

# STEP 21: Prediction

```python
y_pred = classifier.predict(X_test)
```

Purpose:
- Predict Spam/Ham on unseen messages.

---

# STEP 22: Evaluation

Confusion Matrix

```python
confusion_matrix(y_test,y_pred)
```

Accuracy

```python
accuracy_score(y_test,y_pred)
```

Classification Report

```python
classification_report(y_test,y_pred)
```

Metrics:

- Accuracy
- Precision
- Recall
- F1 Score

---

# Final Pipeline

SMS Message
↓
Cleaning
↓
Tokenization
↓
Word2Vec Training
↓
Word Embeddings
↓
Average Word Embeddings
↓
Feature Matrix X
↓
Train/Test Split
↓
Random Forest
↓
GridSearchCV
↓
Prediction
↓
Evaluation

In [1]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 6.2 MB/s eta 0:00:04
   --- ------------------------------------ 1.8/24.4 MB 5.3 MB/s eta 0:00:05
   --- ------------------------------------ 2.1/24.4 MB 5.0 MB/s eta 0:00:05
   ---- ----------------------------------- 2.9/24.4 MB 3.4 MB/s eta 0:00:07
   ------ --------------------------------- 3.7/24.4 MB 3.6 MB/s eta 0:00:06
   ------ --------------------------------- 4.2/24.4 MB 3.7 MB/s eta 0:00:06
   -------- ------------------------------- 5.2/24.4 MB 3.7 MB/s eta 0:00:06
   --------- ------------------------------ 6.0/24.4 MB 3.7 MB/s eta 0:00:05
   ----------- ---------------------------- 6.8/24.4 MB 3.8 MB/s eta 0:00:05
   ------------ --------------------------- 7.6/24.4 MB 3.8 MB/s eta 0:00:05
   ------------- -------------------------- 8.4/24.4 MB 3.8 MB/s eta 0:00:05
   --------------- ------------------------ 9.4/24.4 MB 3.8 MB/s eta 0:00:04
   ---

In [2]:
import gensim
from gensim.models import Word2Vec,KeyedVectors


In [ ]:
import gensim.downloader as api
wv=api.load('word2vec-google-news-300')
vec_king=wv['king']

In [ ]:
vec_king

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# %matplotlib-inline



In [ ]:
messages = pd.read_csv( "/content/SMSSpamCollection.csv",
                        sep ='\t',
                        names = ["label","message"])

In [ ]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize,word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
lemmetizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
# Data cleaning and preprocessing

corpus = []

for i in range(len(messages)):
  review = re.sub("[^a-zA-Z]"," ",messages['message'][i])
  review = review.lower()
  review = review.split()
  review = [lemmetizer.lemmatize(word) for word in review]
  review =" ".join(review)
  corpus.append(review)

In [ ]:
corpus


['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
 'ok lar joking wif u oni',
 'free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry question std txt rate t c s apply over s',
 'u dun say so early hor u c already then say',
 'nah i don t think he go to usf he life around here though',
 'freemsg hey there darling it s been week s now and no word back i d like some fun you up for it still tb ok xxx std chgs to send to rcv',
 'even my brother is not like to speak with me they treat me like aid patent',
 'a per your request melle melle oru minnaminunginte nurungu vettam ha been set a your callertune for all caller press to copy your friend callertune',
 'winner a a valued network customer you have been selected to receivea prize reward to claim call claim code kl valid hour only',
 'had your mobile month or more u r entitled to update to the latest colour mobile with camera for free call the mobile up

In [ ]:
from gensim.utils import simple_preprocess

In [ ]:
words = []

for sentence in corpus:
  sentence_token = sent_tokenize(sentence)
  for sent in sentence_token:
      words.append(simple_preprocess(sent))


print(words)

[['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'great', 'world', 'la', 'buffet', 'cine', 'there', 'got', 'amore', 'wat'], ['ok', 'lar', 'joking', 'wif', 'oni'], ['free', 'entry', 'in', 'wkly', 'comp', 'to', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'to', 'to', 'receive', 'entry', 'question', 'std', 'txt', 'rate', 'apply', 'over'], ['dun', 'say', 'so', 'early', 'hor', 'already', 'then', 'say'], ['nah', 'don', 'think', 'he', 'go', 'to', 'usf', 'he', 'life', 'around', 'here', 'though'], ['freemsg', 'hey', 'there', 'darling', 'it', 'been', 'week', 'now', 'and', 'no', 'word', 'back', 'like', 'some', 'fun', 'you', 'up', 'for', 'it', 'still', 'tb', 'ok', 'xxx', 'std', 'chgs', 'to', 'send', 'to', 'rcv'], ['even', 'my', 'brother', 'is', 'not', 'like', 'to', 'speak', 'with', 'me', 'they', 'treat', 'me', 'like', 'aid', 'patent'], ['per', 'your', 'request', 'melle', 'melle', 'oru', 'minnaminunginte', 'nurungu', 'vettam', 'ha', 'been', 'set', 

### corpus ex:
'go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat'

###

### after sentence tokenization:

['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat']

### after simple preprocess:
['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'great', 'world', 'la', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']

simple_preprocess()

Convert a document into a list of lowercase tokens, ignoring tokens that are too short or too long.

Uses ~gensim.utils.tokenize internally.

In [ ]:
import gensim

### lets train Word2Vec from scratch
## google word2vec vector size = 300

model = gensim.models.Word2Vec(sentences =  words,
                              vector_size=100,
                               epochs=20)

### Word2Vec

def __init__(sentences=None,
 corpus_file=None,
  vector_size=100,
   alpha=0.025,
    window=5,
    min_count=5,
    max_vocab_size=None, sample=0.001, seed=1, workers=3, min_alpha=0.0001, sg=0, hs=0, negative=5, ns_exponent=0.75, cbow_mean=1, hashfxn=hash, epochs=5, null_word=0, trim_rule=None, sorted_vocab=1, batch_words=MAX_WORDS_IN_BATCH, compute_loss=False, callbacks=(), comment=None, max_final_vocab=None, shrink_windows=True)

In [ ]:
model

In [ ]:
# To get all the vocalbulary
model.wv.index_to_key

['to',
 'you',
 'the',
 'and',
 'it',
 'in',
 'is',
 'me',
 'my',
 'for',
 'your',
 'call',
 'of',
 'that',
 'have',
 'on',
 'now',
 'are',
 'can',
 'so',
 'but',
 'not',
 'or',
 'we',
 'do',
 'get',
 'at',
 'ur',
 'if',
 'will',
 'be',
 'with',
 'no',
 'just',
 'this',
 'gt',
 'lt',
 'go',
 'how',
 'up',
 'when',
 'day',
 'ok',
 'what',
 'free',
 'from',
 'all',
 'out',
 'know',
 'll',
 'come',
 'like',
 'good',
 'time',
 'am',
 'then',
 'got',
 'wa',
 'there',
 'he',
 'love',
 'text',
 'only',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'by',
 'going',
 'don',
 'stop',
 'she',
 'home',
 'about',
 'lor',
 'sorry',
 'see',
 'still',
 'mobile',
 'take',
 'back',
 'da',
 'reply',
 'dont',
 'our',
 'think',
 'tell',
 'week',
 'phone',
 'hi',
 'new',
 'they',
 'later',
 'please',
 'any',
 'pls',
 'her',
 'ha',
 'did',
 'co',
 'msg',
 'been',
 'min',
 'an',
 'some',
 'dear',
 'night',
 'make',
 'who',
 'here',
 'message',
 'well',
 'say',
 'where',
 're',
 'thing',
 'much',
 'hope

In [ ]:
# vocabulary size
model.corpus_count

5569

In [ ]:
model.epochs
# more epochs better result

20

In [ ]:
model.wv.similar_by_word("good")
# cosine similarity in the corpus

[('morning', 0.8661189079284668),
 ('evening', 0.8542566895484924),
 ('sweet', 0.8467262983322144),
 ('great', 0.826341450214386),
 ('hope', 0.8046209216117859),
 ('dear', 0.8015139698982239),
 ('sleep', 0.7989289164543152),
 ('nice', 0.778820812702179),
 ('wonderful', 0.7742507457733154),
 ('afternoon', 0.7692381143569946)]

In [ ]:
model.wv["good"]
# good is converted to 100 dimensional vectors

array([-1.2453079 ,  0.66679305, -0.31236327,  0.08682911,  0.17281596,
        0.1751783 ,  0.65349895,  0.7952173 , -0.4187512 , -0.08522771,
        0.18031502, -1.2840852 ,  0.9936791 ,  0.44695657, -0.21307214,
       -0.15935172,  0.6032621 , -0.30668983, -0.11787516, -1.0311373 ,
        1.3174459 ,  0.8330387 ,  0.38453844, -0.3363081 , -0.03729518,
        0.73534733,  0.3138079 , -0.1107387 , -0.45015767, -0.17483968,
       -0.06808946, -0.042284  ,  0.12666468, -0.4848126 , -0.0104383 ,
        0.41698617, -0.13315365,  0.05683832,  0.7917015 ,  0.1037783 ,
        0.26692197, -0.09905123,  0.24246737,  0.4737391 , -0.30356127,
        0.05862286, -0.6655505 , -0.22949336, -0.7601103 , -0.1117182 ,
        1.5937034 , -0.1107268 , -0.46967208, -0.9639055 ,  1.0392064 ,
       -0.22026493,  1.1232222 ,  0.29541597, -0.39239067, -0.34902224,
       -0.75701743, -0.24586579, -0.69252217, -0.563986  , -1.5327749 ,
       -0.02154156, -0.09782276,  0.34533986, -0.42090997,  0.55

In [ ]:
model.wv["good"].shape
#  vector size for good

(100,)

In [ ]:
words

[['go',
  'until',
  'jurong',
  'point',
  'crazy',
  'available',
  'only',
  'in',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'there',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'in',
  'wkly',
  'comp',
  'to',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'to',
  'to',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply',
  'over'],
 ['dun', 'say', 'so', 'early', 'hor', 'already', 'then', 'say'],
 ['nah',
  'don',
  'think',
  'he',
  'go',
  'to',
  'usf',
  'he',
  'life',
  'around',
  'here',
  'though'],
 ['freemsg',
  'hey',
  'there',
  'darling',
  'it',
  'been',
  'week',
  'now',
  'and',
  'no',
  'word',
  'back',
  'like',
  'some',
  'fun',
  'you',
  'up',
  'for',
  'it',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'to',
  'send',
  'to',
  'rcv'],
 ['even',
  'my',
  'brother',
  'is',
  'not',
  'like',
  'to',
  'spea

In [ ]:
words[0]
# now each word in words[0] or first sentence creates vectors of 100 dimension for each word

['go',
 'until',
 'jurong',
 'point',
 'crazy',
 'available',
 'only',
 'in',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'there',
 'got',
 'amore',
 'wat']

In [ ]:
def avg_word2vec(doc):
  # remove out-of-vocabulary words
  # sent = [word for word in doc if word in model.wv.index_to_key]
  # print(sent)

  """
  row wise avereging
  first matrix addition of each words vector in sentence then avg
  """
  # Filter out words not in the model's vocabulary
  word_vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]

  # If no words from the sentence are in the vocabulary, return a zero vector
  if not word_vectors:
    return np.zeros(model.vector_size) # Use model.vector_size for the dimension

  return np.mean(word_vectors, axis=0)

In [ ]:
!pip install tqdm

from tqdm import tqdm

# for progression tracking

In [ ]:
# apply for all the sentences
sentence_vector =[]

for i in tqdm(range(len(words))):
  avg_vec = avg_word2vec(words[i])
  sentence_vector.append(avg_vec)

100%|██████████| 5569/5569 [00:01<00:00, 2894.51it/s]


In [ ]:
sentence_vector

# list of sentence vectors

[array([-0.09577187,  0.1033846 ,  0.04357827,  0.02453117,  0.0473702 ,
        -0.41553065,  0.1581168 ,  0.3289226 , -0.2165454 ,  0.09778827,
        -0.22159907, -0.23941821, -0.24142434, -0.0049979 ,  0.15811844,
        -0.12274398,  0.05809852, -0.15663351, -0.0538326 , -0.395655  ,
         0.0487798 ,  0.17075801,  0.11998334, -0.09629162, -0.1836478 ,
        -0.14740144, -0.40854886, -0.22389032, -0.3301532 ,  0.0483329 ,
         0.33567387, -0.05006414,  0.05579742, -0.24697235, -0.11664221,
         0.26262534, -0.06645869, -0.01486378, -0.16600223, -0.37099713,
         0.22080256, -0.18980527, -0.10630617,  0.07653126,  0.26949036,
        -0.10427107, -0.07682417, -0.00682772,  0.2166357 ,  0.15033141,
         0.17629363, -0.0696864 , -0.01172005,  0.21414074, -0.17629863,
         0.11241315,  0.33207446,  0.12144077, -0.32108814,  0.00530139,
        -0.0434435 ,  0.1443463 , -0.0972408 , -0.12442276, -0.23822156,
         0.09975364,  0.1454761 ,  0.16276854, -0.3

In [ ]:
len(sentence_vector)

5569

In [ ]:
sentence_vector[0].shape

(100,)

# independent feature

In [95]:
X = np.array(sentence_vector)
X.shape

(5569, 100)

In [96]:
X.size

556900

In [97]:
messages.shape
# intial record 5572 and now 5569 ... 3 records missing

(5572, 2)

# dependent features

In [119]:
y_preprocessed= pd.get_dummies(messages['label'])
y_preprocessed =y_preprocessed.iloc[:,0].values

In [120]:
y_preprocessed.shape

(5572,)

## something fishhy

In [121]:

# intial record 5572 and now 5569 ... 3 records missing

missing = [
    (idx, original)
    for idx, (processed, original)
    in enumerate(zip(corpus, messages['message']))
    if len(processed.strip()) == 0
]

print(missing)

[(1612, '645'), (3376, ':) '), (4824, ':-) :-)')]


In [ ]:
# Original rows = 5572
# Processed rows = 5569
# Missing = 3

In [125]:
## need  to find the y valaues corresonded to those indexes
# to balance the shape of input features (x,y)

y = messages[list (map(lambda x : len(x)> 0,corpus))]
y = pd.get_dummies(y['label'])
y =y.iloc[:,0].values

In [126]:
y.shape

(5569,)

In [128]:
y

array([ True,  True, False, ...,  True,  True,  True])

### some more preprocessing for independent features

In [131]:
X.shape

(5569, 100)

In [133]:
X[0].shape
# X[0] is first sentence vector

(100,)

In [135]:
X[0]

array([-0.09577187,  0.1033846 ,  0.04357827,  0.02453117,  0.0473702 ,
       -0.41553065,  0.1581168 ,  0.3289226 , -0.2165454 ,  0.09778827,
       -0.22159907, -0.23941821, -0.24142434, -0.0049979 ,  0.15811844,
       -0.12274398,  0.05809852, -0.15663351, -0.0538326 , -0.39565501,
        0.0487798 ,  0.17075801,  0.11998334, -0.09629162, -0.1836478 ,
       -0.14740144, -0.40854886, -0.22389032, -0.3301532 ,  0.0483329 ,
        0.33567387, -0.05006414,  0.05579742, -0.24697235, -0.11664221,
        0.26262534, -0.06645869, -0.01486378, -0.16600223, -0.37099713,
        0.22080256, -0.18980527, -0.10630617,  0.07653126,  0.26949036,
       -0.10427107, -0.07682417, -0.00682772,  0.2166357 ,  0.15033141,
        0.17629363, -0.0696864 , -0.01172005,  0.21414074, -0.17629863,
        0.11241315,  0.33207446,  0.12144077, -0.32108814,  0.00530139,
       -0.0434435 ,  0.1443463 , -0.0972408 , -0.12442276, -0.23822156,
        0.09975364,  0.1454761 ,  0.16276854, -0.34607109,  0.12

In [134]:
X[0].reshape(1,-1)

# before : array([-0.09577187,  0.1033846 ,  0.04357827, .....
# shape: (100,)

# after : array([[-0.09577187,  0.1033846 ,  0.04357827,  .....
# shape : (1,100)

# is it like transpose?

array([[-0.09577187,  0.1033846 ,  0.04357827,  0.02453117,  0.0473702 ,
        -0.41553065,  0.1581168 ,  0.3289226 , -0.2165454 ,  0.09778827,
        -0.22159907, -0.23941821, -0.24142434, -0.0049979 ,  0.15811844,
        -0.12274398,  0.05809852, -0.15663351, -0.0538326 , -0.39565501,
         0.0487798 ,  0.17075801,  0.11998334, -0.09629162, -0.1836478 ,
        -0.14740144, -0.40854886, -0.22389032, -0.3301532 ,  0.0483329 ,
         0.33567387, -0.05006414,  0.05579742, -0.24697235, -0.11664221,
         0.26262534, -0.06645869, -0.01486378, -0.16600223, -0.37099713,
         0.22080256, -0.18980527, -0.10630617,  0.07653126,  0.26949036,
        -0.10427107, -0.07682417, -0.00682772,  0.2166357 ,  0.15033141,
         0.17629363, -0.0696864 , -0.01172005,  0.21414074, -0.17629863,
         0.11241315,  0.33207446,  0.12144077, -0.32108814,  0.00530139,
        -0.0434435 ,  0.1443463 , -0.0972408 , -0.12442276, -0.23822156,
         0.09975364,  0.1454761 ,  0.16276854, -0.3

In [136]:
X[0].reshape(1,-1).shape

(1, 100)

### `reshape(1, -1)` Short Note

> **`reshape(1, -1)` converts a 1D array into a single-row 2D array, commonly used in machine learning when predicting on one sample.**

Example:

```python
X[0]          # Shape: (100,)
X[0].reshape(1,-1)   # Shape: (1,100)
```

```text
(100,)    →  1D Array
     ↓
reshape(1,-1)
     ↓
(1,100)   → 1 sample, 100 features
```

### Memory Trick

```text
reshape(1,-1)  → One row, many columns
reshape(-1,1)  → Many rows, one column
```

### Interview One-Liner

> `reshape(1,-1)` is used to convert a feature vector into the format `(n_samples, n_features)` required by most scikit-learn models.


In [138]:
# for all the sentence vetor
#  conversion into dataframe

df = pd.DataFrame()
df # empty dataframe

""


In [142]:
# for i in range(len(X)):
#   df = df.append(pd.DataFrame(X[i].reshape(1,-1)),ignore_index=True)

AttributeError: 'DataFrame' object has no attribute 'append'

pandas 2 remove append

In [143]:
df = pd.DataFrame(X)

In [144]:
X.shape

(5569, 100)

In [145]:
df.shape

(5569, 100)

In [148]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.095772,0.103385,0.043578,0.024531,0.047370,-0.415531,0.158117,0.328923,-0.216545,0.097788,...,0.167308,0.249446,0.113433,0.133669,0.469819,0.255544,0.137110,-0.063313,0.032301,-0.064095
1,0.025158,0.041011,-0.294899,0.305691,0.015330,-0.511339,0.109748,0.610623,-0.137346,0.545079,...,0.094702,0.053300,0.019214,0.415784,0.326328,0.266167,0.049836,-0.084251,-0.405472,-0.134726
2,0.035245,0.303708,0.327549,0.281500,-0.013044,-1.003542,0.166613,0.167967,-0.430605,-0.468959,...,-0.076878,0.324716,0.028388,0.101823,0.425306,0.099065,-0.426069,-0.529767,0.553557,-0.069057
3,-0.291784,-0.065895,-0.353062,0.209366,0.007736,-0.447913,0.168337,0.617921,-0.477683,0.459306,...,0.127631,0.050088,0.029251,0.541452,0.089072,0.404683,0.152267,-0.286471,-0.306267,0.046427
4,-0.201949,0.175583,-0.151708,-0.249232,0.070822,-0.251440,0.086750,0.509568,-0.492084,-0.354571,...,0.321433,0.085035,0.287211,0.176050,0.165468,0.445315,0.075037,-0.196075,0.075031,-0.158275


### Final independent feature

In [149]:
X = df

In [150]:
X

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.095772,0.103385,0.043578,0.024531,0.047370,-0.415531,0.158117,0.328923,-0.216545,0.097788,...,0.167308,0.249446,0.113433,0.133669,0.469819,0.255544,0.137110,-0.063313,0.032301,-0.064095
1,0.025158,0.041011,-0.294899,0.305691,0.015330,-0.511339,0.109748,0.610623,-0.137346,0.545079,...,0.094702,0.053300,0.019214,0.415784,0.326328,0.266167,0.049836,-0.084251,-0.405472,-0.134726
2,0.035245,0.303708,0.327549,0.281500,-0.013044,-1.003542,0.166613,0.167967,-0.430605,-0.468959,...,-0.076878,0.324716,0.028388,0.101823,0.425306,0.099065,-0.426069,-0.529767,0.553557,-0.069057
3,-0.291784,-0.065895,-0.353062,0.209366,0.007736,-0.447913,0.168337,0.617921,-0.477683,0.459306,...,0.127631,0.050088,0.029251,0.541452,0.089072,0.404683,0.152267,-0.286471,-0.306267,0.046427
4,-0.201949,0.175583,-0.151708,-0.249232,0.070822,-0.251440,0.086750,0.509568,-0.492084,-0.354571,...,0.321433,0.085035,0.287211,0.176050,0.165468,0.445315,0.075037,-0.196075,0.075031,-0.158275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5564,-0.086657,0.383521,0.601003,0.111542,0.125144,-0.653620,0.527887,0.042857,-0.382801,-0.596172,...,0.102581,0.145620,0.031034,-0.398123,0.552993,0.063925,-0.094906,-0.454986,0.773292,-0.060037
5565,-0.126553,-0.300490,-0.162667,0.140546,0.186034,-0.505292,0.363896,0.738264,-0.485737,0.142521,...,0.484031,0.181741,-0.240864,0.342563,0.546960,0.590369,-0.282111,0.090745,-0.147441,0.023276
5566,-0.284125,0.301749,-0.004018,-0.419322,-0.177661,-0.107267,-0.019384,0.388894,-0.248639,-0.278979,...,0.374848,0.230788,0.345147,0.195773,0.350057,0.364066,0.235663,0.162685,0.100180,-0.160058
5567,-0.191318,0.261013,-0.037070,-0.252051,-0.031511,-0.271087,0.054020,0.329073,-0.371308,-0.423097,...,0.254881,0.211762,0.393979,0.163977,0.292287,0.407055,0.007878,-0.053910,0.166246,-0.160974


# Train test split

In [151]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)


In [152]:
X_train

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
253,-0.985898,0.347522,0.286280,-0.600071,0.623272,-0.600092,-0.002977,0.931808,-0.629798,-0.772258,...,1.191521,-0.049388,0.020667,-0.066891,-0.161518,0.913743,-0.006598,-0.754854,0.014389,0.204658
1114,-0.165652,0.216988,-0.009334,-0.150035,-0.130111,-0.302628,0.273293,0.314761,-0.293985,-0.364423,...,0.166237,0.174114,0.382288,0.134358,0.280822,0.380636,0.312320,0.037212,0.206532,-0.223292
4613,-0.309019,0.148127,-0.088906,-0.020721,0.137751,-0.145163,0.142704,0.504079,-0.206497,0.151383,...,0.366931,0.056647,0.041999,0.070454,0.338684,0.409068,0.185536,0.008372,0.111491,-0.101426
4376,-0.618247,0.262800,0.121258,-0.187543,0.261065,-0.337623,0.333467,0.467508,-0.554213,-0.233900,...,0.432008,0.232539,0.118755,0.227831,-0.044267,0.629869,0.397174,-0.309912,0.160392,-0.111360
1012,-0.225065,0.114473,-0.070063,-0.168404,0.185566,-0.485979,0.139119,0.600369,-0.425022,-0.289080,...,0.676892,0.018967,-0.010338,-0.124320,0.045066,0.499419,-0.088481,-0.176506,0.073016,-0.106999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4931,0.200177,0.699329,0.221508,0.180707,0.274841,-0.751506,0.017682,0.607115,-0.368628,-0.591962,...,0.495569,-0.051528,-0.413165,-0.164812,0.521108,-0.113637,-0.603512,-0.416658,0.030738,-0.540362
3264,0.052498,0.280502,0.065701,-0.167859,-0.006519,-0.576910,0.398578,0.613325,-0.544669,-0.641734,...,0.182698,0.109724,0.212356,0.302708,0.384426,0.397738,0.155356,0.112133,0.379040,-0.324854
1653,-0.161668,-0.001093,-0.084494,0.085803,0.098552,-0.317379,0.074018,0.480456,-0.234116,0.216436,...,0.306973,0.127091,0.041363,0.234070,0.346026,0.316727,-0.104281,-0.124859,-0.137675,-0.058034
2607,-0.316296,0.349706,0.027066,-0.471794,0.084822,-0.211647,0.039697,0.530758,-0.310137,-0.626204,...,0.562788,-0.186609,0.243886,-0.002476,0.160292,0.423051,0.147104,-0.179815,0.015096,0.067541


In [153]:
y_train

array([ True,  True,  True, ...,  True,  True,  True])

## NULL ,NAN checking

In [155]:
X.isnull().sum()

,0
0,0
1,0
2,0
3,0
4,0
...,...
95,0
96,0
97,0
98,0


## dense vectors -> best model ->random forest classifier

In [154]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [156]:
classifier.fit(X_train,y_train)

RandomForestClassifier()

# hyperparameter tuning

In [175]:
para ={
    "n_estimators" : [100,200,300,400,500],
    "max_depth" : [10,20,30,40,50]
}

In [176]:
from sklearn.model_selection import GridSearchCV
model = GridSearchCV(estimator = classifier,
                  param_grid = para,
                  n_jobs =-1,
                  cv =5,
                  verbose =3)

In [ ]:
model.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


# testing and predict

In [162]:
y_pred = classifier.predict(X_test)

# metrics

In [163]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[144  10]
 [  5 955]]
0.9865350089766607
              precision    recall  f1-score   support

       False       0.97      0.94      0.95       154
        True       0.99      0.99      0.99       960

    accuracy                           0.99      1114
   macro avg       0.98      0.96      0.97      1114
weighted avg       0.99      0.99      0.99      1114

